# Real-Time Feedforward Neural Network Classifier (Simplified)

Train a small FFNN once on labeled data, then simulate a live stream of new points arriving one at a time and classify each one instantly, updating a plot as points come in.

## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

torch.manual_seed(0)
np.random.seed(0)

## 2. Define a Small Feedforward Network

A single hidden layer network: `Linear(2, 8) -> ReLU -> Linear(8, 1) -> Sigmoid`

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid(),
)

## 3. Train Once, Offline

Train on a labeled dataset (like training a spam/anomaly detector before deploying it).

In [ ]:
X_train, y_train = make_classification(
    n_samples=200, n_features=2, n_redundant=0,
    n_clusters_per_class=1, random_state=0
)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
criterion = nn.BCELoss()

for epoch in range(150):
    optimizer.zero_grad()
    loss = criterion(model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
print(f"Training done. Final loss: {loss.item():.4f}")

## 4. Simulate a Stream of New, Unseen Points

These points were never seen during training - like new sensor readings or transactions arriving live.

In [ ]:
X_stream, y_stream = make_classification(
    n_samples=30, n_features=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)

## 5. Classify Points One at a Time and Plot Live

Each point is classified the moment it "arrives", plotted immediately, and the running accuracy is printed. In Jupyter, the plot updates inline as the loop runs.

In [ ]:
%matplotlib inline
from IPython.display import clear_output

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(X_stream[:, 0].min() - 1, X_stream[:, 0].max() + 1)
ax.set_ylim(X_stream[:, 1].min() - 1, X_stream[:, 1].max() + 1)

correct = 0
seen_x, seen_y, seen_c = [], [], []

for i, (point, true_label) in enumerate(zip(X_stream, y_stream)):
    with torch.no_grad():
        prob = model(torch.tensor(point, dtype=torch.float32)).item()
    pred_label = 1 if prob >= 0.5 else 0
    correct += int(pred_label == true_label)

    color = "red" if pred_label == 1 else "blue"
    seen_x.append(point[0])
    seen_y.append(point[1])
    seen_c.append(color)

    running_acc = correct / (i + 1)
    print(f"Point {i+1}/{len(X_stream)} | Predicted: {pred_label} "
          f"(confidence {prob:.2f}) | Actual: {true_label} | "
          f"Running accuracy: {running_acc*100:.1f}%")

    clear_output(wait=True)
    ax.clear()
    ax.set_title("Real-Time FFNN Classifier")
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.set_xlim(X_stream[:, 0].min() - 1, X_stream[:, 0].max() + 1)
    ax.set_ylim(X_stream[:, 1].min() - 1, X_stream[:, 1].max() + 1)
    ax.scatter(seen_x, seen_y, c=seen_c, edgecolors="black", s=80)
    display(fig)

## 6. Save Final Snapshot

In [ ]:
fig.savefig("realtime_result.png")
print("Final snapshot saved as realtime_result.png")